In [ ]:
import __main__
import sys, os
project_root = os.path.abspath("..")  # adjust if notebook is elsewhere
sys.path.insert(0, project_root)
from typing import Dict, List, Literal, Tuple, Optional, Any
import logging
import math
import gc
import time
import json
import random
from pathlib import Path

import category_encoders as ce
from matplotlib import pyplot as plt
import numexpr as ne # makes numpy operations faster
import numpy as np
import pandas as pd
from tqdm import tqdm
from mpl_toolkits.mplot3d import Axes3D

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, accuracy_score, mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.random_projection import GaussianRandomProjection

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    # print(torch.cuda.memory_reserved(0) / 1e6, "MB reserved")
    # print(torch.cuda.memory_allocated(0) / 1e6, "MB allocated")

import param_config.config_paths as P

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.info("Starting process...")
logging.warning("Something looks off...")
logging.error("Something failed.")


In [ ]:
# step 0: load file
file_loc    = "/Users/fouadabiad/Downloads/PAMAP2_Dataset/Protocol"
record_name = "subject108.dat"
record_path = f"{file_loc}/{record_name}"
df          = pd.read_csv(record_path, sep=' ', header=None)

df = df.bfill().ffill()  # fill NaNs for all columns immediately
df = df.sample(frac=0.01, random_state=42)


In [ ]:
# step 1: split features
row = df.values  # shape (num_rows, num_features)

# 1️⃣ Sphere block: 3D vectors (accel + gyro)
sphere_cols = [
    slice(1,4),  slice(7,10),   # hand
    slice(20,23), slice(27,30), # chest
    slice(37,40), slice(44,47)]  # ankle
sphere_vectors  = [row[:, s] for s in sphere_cols]
Z_sphere        = np.concatenate(sphere_vectors, axis=1)  # shape (num_rows, 18)
Z_sphere_scaled = StandardScaler().fit_transform(Z_sphere)

# 2️⃣ Torus block: cyclic/stride phase (mocked here as random)
Z_torus         = np.random.rand(row.shape[0], 1) * 2*np.pi  # shape (num_rows, 1)
Z_torus_scaled  = (Z_torus - np.pi) / np.pi  # scale to [-1,1]

# 3️⃣ Euclidean block: scalar features
euclid_cols     = [2, 3, 20, 37]  # Python indexing
Z_euclid        = row[:, euclid_cols]  # shape (num_rows, 4)
Z_euclid_scaled = StandardScaler().fit_transform(Z_euclid)

# Combine all features into one Euclidean input
Z_all = np.concatenate([Z_sphere_scaled, Z_euclid_scaled], axis=1) # for full Euclidean set

print("Sphere block shape:", Z_sphere.shape)
# print("Torus block shape:", Z_torus.shape)
print("Euclidean block shape:", Z_euclid.shape)
print("ALL shape:", Z_all.shape)


In [ ]:
"(V)AE"
# 1️⃣ Euclidean Autoencoder
class EuclideanAE(nn.Module):
    def __init__(self, input_dim:int, hidden_dim:int, latent_dim:int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim))
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim))
    def forward(self, x:torch.Tensor):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return z, x_hat

# 2️⃣ Spherical Autoencoder (vMF prior); uses L2 norm to project to hypersphere
class SphericalAE(nn.Module):
    def __init__(self, input_dim:int, hidden_dim:int, latent_dim:int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim))

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim))
    def forward(self, x:torch.Tensor):
        z_raw = self.encoder(x)
        norms = torch.norm(z_raw, p=2, dim=1, keepdim=True) + 1e-8
        z     = z_raw / norms
        x_hat = self.decoder(z)
        return z, x_hat

# 3️⃣ Torus Autoencoder (wrap-aware); encode phase as sin/cos, decode same
class TorusAE(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, latent_dim: int):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),)   # ⬅️ k angles
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),)

    def forward(self, x):
        theta = self.encoder(x)                  # (N, k)
        x_hat = self.decoder(theta)
        return theta, x_hat

def train_ae(
    model: nn.Module,
    x: torch.Tensor,
    epochs: int = 50,
    lr: float = 1e-3,
    reg_fn=None,
    reg_weight: float = 0.0,):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        optimizer.zero_grad()
        z, x_hat = model(x)
        loss     = F.mse_loss(x_hat, x)
        if reg_fn is not None:
            loss = loss + reg_weight * reg_fn(z)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if epoch % 10 == 0 or epoch == epochs-1:
            print(f"Epoch {epoch+1}/{epochs}, loss: {loss.item():.6f}")
    return model


In [ ]:
# step 2: encode to z
x_sphere = torch.tensor(Z_sphere_scaled, dtype=torch.float32)  # (num_rows, 18)
x_torus  = torch.tensor(Z_torus_scaled, dtype=torch.float32)   # (num_rows, 1)
x_euclid = torch.tensor(Z_euclid_scaled, dtype=torch.float32) # (num_rows, 4)
x_all    = torch.tensor(Z_all, dtype=torch.float32) # for full Euclidean set

# --- instantiate encoders ---
sphere_input      = Z_sphere.shape[1]
sphere_latent_dim = 3
sphere_hidden_dim = 16
sphere_ae         = SphericalAE(sphere_input, sphere_hidden_dim, sphere_latent_dim)

torus_input       = Z_torus.shape[1]
latent_dim_torus  = 2
torus_hidden_dim  = 8
torus_ae          = TorusAE(torus_input, torus_hidden_dim, latent_dim_torus)

euclid_input      = Z_euclid.shape[1]
euclid_latent_dim = 2
euclid_hidden_dim = 8
euclid_ae         = EuclideanAE(euclid_input, euclid_hidden_dim, euclid_latent_dim)

# full euclidean set
input_dim  = Z_all.shape[1]
hidden_dim = 16
latent_dim = sphere_latent_dim + euclid_latent_dim
euclid_full_ae = EuclideanAE(input_dim, hidden_dim, latent_dim)

# --- train ---
epochs         = 100
euclid_ae      = train_ae(euclid_ae, x_euclid, epochs=epochs)
sphere_ae      = train_ae(sphere_ae, x_sphere, epochs=epochs)
euclid_full_ae = train_ae(euclid_full_ae, x_all, epochs=epochs)

def torus_uniform_regularizer(theta):
    return (theta.cos().mean()**2 + theta.sin().mean()**2)

torus_ae = train_ae(
    torus_ae,
    x_torus,
    epochs=epochs,
    reg_fn=torus_uniform_regularizer,
    reg_weight=0.1,)

# --- encode ---
z_sphere_enc, _ = sphere_ae(x_sphere)
z_torus_enc,  _ = torus_ae(x_torus)
z_euclid_enc, _ = euclid_ae(x_euclid)

z_euclid_full_enc, _ = euclid_full_ae(x_all)
z_euclid_full_np     = z_euclid_full_enc.detach().numpy()
print("Full Euclidean AE latent shape:", z_euclid_full_np.shape)

# --- concatenate into final hybrid latent vectors ---
# z_hybrid_encoded    = torch.cat([z_sphere_enc, z_torus_enc, z_euclid_enc], dim=1)  # (num_rows, total_latent_dim)
z_hybrid_encoded    = torch.cat([z_sphere_enc, z_euclid_enc], dim=1)  # (num_rows, total_latent_dim)
z_hybrid_encoded_np = z_hybrid_encoded.detach().numpy()
print("Encoded hybrid latent vectors shape:", z_hybrid_encoded.shape)

print("Sphere latent NaNs:", np.isnan(z_sphere_enc.detach().numpy()).sum())
print("Torus latent NaNs:", np.isnan(z_torus_enc.detach().numpy()).sum())
print("Euclid latent NaNs:", np.isnan(z_euclid_enc.detach().numpy()).sum())


In [ ]:
"plot"
z_s = z_sphere_enc.detach().numpy()  # (num_rows, 3)
fig = plt.figure()
ax  = fig.add_subplot(111, projection='3d')
ax.scatter(z_s[:,0], z_s[:,1], z_s[:,2], c='blue', s=5)
ax.set_title("Sphere latent space")
plt.show()

z_t = z_torus_enc.detach().numpy()  # (num_rows, 2)
plt.figure()
plt.scatter(np.cos(z_t[:,0]), np.sin(z_t[:,0]), c='red', s=5, label='theta1')
plt.scatter(np.cos(z_t[:,1]), np.sin(z_t[:,1]), c='green', s=5, label='theta2')
plt.axis('equal')
plt.title("Torus latent space")
plt.legend()
plt.show()

z_e = z_euclid_enc.detach().numpy()  # (num_rows, 2)
plt.scatter(z_e[:,0], z_e[:,1], c='purple', s=5)
plt.title("Euclidean latent space")
plt.show()

Z_hybrid = z_hybrid_encoded_np  # (num_rows, total_latent_dim)
Z_2d = PCA(n_components=2).fit_transform(Z_hybrid)
plt.scatter(Z_2d[:,0], Z_2d[:,1], s=5)
plt.title("Hybrid latent space (PCA 2D)")
plt.show()


In [ ]:
# step 3: hybrid distance (sphere + euclidean)
latent_dim_sphere = z_sphere_enc.shape[1]
latent_dim_euclid = z_euclid_enc.shape[1]

idx_sphere = slice(0, latent_dim_sphere)
idx_euclid = slice(latent_dim_sphere, None)

Z = z_hybrid_encoded_np
num_rows = Z.shape[0]

dist_matrix = np.zeros((num_rows, num_rows))

for i in range(num_rows):
    u   = Z[i]
    u_s = u[idx_sphere]
    u_s = u_s / np.linalg.norm(u_s)

    for j in range(i, num_rows):
        v = Z[j]

        # sphere
        v_s = v[idx_sphere]
        v_s = v_s / np.linalg.norm(v_s)
        d_sphere = np.arccos(np.clip(np.dot(u_s, v_s), -1.0, 1.0))

        # euclidean
        u_e, v_e = u[idx_euclid], v[idx_euclid]
        d_euclid = np.linalg.norm(u_e - v_e)

        d = d_sphere + d_euclid
        dist_matrix[i, j] = d
        dist_matrix[j, i] = d


In [ ]:
# torus check
theta = z_torus_enc.detach().cpu().numpy().ravel()

print("mean cos:", np.mean(np.cos(theta)))
print("mean sin:", np.mean(np.sin(theta)))
plt.hist(theta % (2*np.pi), bins=50)
plt.show()


In [ ]:
"new metrics"
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, r2_score

def propagate_knn(Z_latent, y_true, missing_frac=0.2, k=15, tau=1.0):
    """Perform kNN label propagation on latent vectors."""
    num_rows = Z_latent.shape[0]
    dist_matrix = np.zeros((num_rows, num_rows))
    
    # compute L2 distances
    for i in range(num_rows):
        u = Z_latent[i]
        for j in range(i, num_rows):
            v = Z_latent[j]
            d = np.linalg.norm(u - v)
            dist_matrix[i, j] = d
            dist_matrix[j, i] = d

    # mask random fraction
    num_missing = int(missing_frac * num_rows)
    missing_idx = np.random.choice(num_rows, size=num_missing, replace=False)
    y_input     = y_true.copy().astype(float)
    y_input[missing_idx] = np.nan

    # kNN neighbors
    neighbors_idx = np.argsort(dist_matrix, axis=1)[:, 1:k+1]

    # edge weights
    def compute_weights(dists):
        w = np.exp(-dists / tau)
        return w / w.sum()

    # propagate labels
    y_prop = y_input.copy()
    for i in range(num_rows):
        if np.isnan(y_input[i]):
            neigh     = neighbors_idx[i]
            neigh_y   = y_input[neigh]
            mask      = ~np.isnan(neigh_y)
            if np.sum(mask) == 0:
                continue
            neigh_y   = neigh_y[mask]
            neigh_d   = dist_matrix[i, neigh][mask]
            w         = compute_weights(neigh_d)
            y_prop[i] = np.sum(w * neigh_y)

    return y_prop, missing_idx

def evaluate_metrics(y_true, y_pred, missing_idx):
    """Compute RMSE, MAE, Accuracy, Spearman correlation, and R² for masked points."""
    y_true_masked = y_true[missing_idx]
    y_pred_masked = y_pred[missing_idx]
    mask_valid    = ~np.isnan(y_pred_masked)
    
    if mask_valid.sum() == 0:
        return {k: np.nan for k in ["rmse","mae","acc","spearman","r2"]}
    
    y_true_valid = y_true_masked[mask_valid]
    y_pred_valid = y_pred_masked[mask_valid]


    def safe_spearmanr(x, y):
        if np.std(x) == 0 or np.std(y) == 0:
            return np.nan
        return spearmanr(x, y).correlation

    metrics = {
        "rmse": np.sqrt(mean_squared_error(y_true_valid, y_pred_valid)),
        "mae": mean_absolute_error(y_true_valid, y_pred_valid),
        "acc": accuracy_score(y_true_valid, np.round(y_pred_valid).astype(int)),
        "spearman": safe_spearmanr(y_true_valid, y_pred_valid),
        "r2": r2_score(y_true_valid, y_pred_valid)
    }
    return metrics

def format_metrics(metrics: dict) -> dict:
    """Format all float metrics to 3 decimal places, keep nan as-is."""
    return {k: (f"{v:.3f}" if isinstance(v, float) and not np.isnan(v) else v)
            for k, v in metrics.items()}

# --- Hybrid AE ---
y_prop_hybrid, missing_idx = propagate_knn(z_hybrid_encoded_np, df.iloc[:,1].values)
metrics_hybrid = evaluate_metrics(df.iloc[:,1].values, y_prop_hybrid, missing_idx)
print("Hybrid metrics:", format_metrics(metrics_hybrid))

# --- Full Euclidean AE ---
y_prop_euc, missing_idx = propagate_knn(z_euclid_full_np, df.iloc[:,1].values)
metrics_euc = evaluate_metrics(df.iloc[:,1].values, y_prop_euc, missing_idx)
print("Full Euclidean metrics:", format_metrics(metrics_euc))

# --- Baseline (mean predictor) ---
y_true      = df.iloc[:,1].values
num_rows    = len(y_true)
num_missing = int(0.2 * num_rows)
np.random.seed(42)
missing_idx = np.random.choice(num_rows, size=num_missing, replace=False)

y_baseline = y_true.copy().astype(float)
y_baseline[missing_idx] = np.nan
mean_label = np.nanmean(y_baseline)
y_baseline[np.isnan(y_baseline)] = mean_label

metrics_baseline = evaluate_metrics(y_true, y_baseline, missing_idx)
print("Baseline metrics:", format_metrics(metrics_baseline))
